# ***Data Tour 2026***

### Importation

In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

from imblearn.over_sampling import BorderlineSMOTE

import warnings
warnings.filterwarnings("ignore")

### Chargement

In [2]:
# ============================================================================
# HELPER
# ============================================================================
def safe_read_csv(filepath):
    try:
        return pd.read_csv(filepath, sep=',', encoding='utf-8')
    except:
        return pd.read_csv(filepath, sep=';', encoding='latin-1')

# ============================================================================
# 1. CHARGEMENT
# ============================================================================
data_path = "/kaggle/input/datasets/alimisara/data-datatour2026"

print("Chargement...")
train = safe_read_csv(os.path.join(data_path, "train.csv"))
test  = safe_read_csv(os.path.join(data_path, "test.csv"))

print(f"Train : {train.shape[0]} lignes × {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]} lignes × {test.shape[1]} colonnes")

Chargement...
Train : 1290081 lignes × 11 colonnes
Test  : 430100 lignes × 10 colonnes


### EDA

In [3]:
train.head(10)

,id,period,operation,amount,origin_account,origin_balance_before,origin_balance_after,destination_account,destination_balance_before,destination_balance_after,fraud_flag
0,dtf_0000001_ffa5beb5,0,op_05,636.75,acc_o_307358626ad66fed,87.00,-549.75,acc_d_7fac3b16af7d127b,630.88,1267.62,0
1,dtf_0000002_61992e82,0,op_05,636.12,acc_o_aeb690c57bf5d1de,76.93,76.93,acc_d_1d6120e8b117aa14,731.70,731.70,0
2,dtf_0000003_9a123b6d,0,op_05,681.00,acc_o_655c41913944d2b7,15943.74,15262.75,acc_d_ec2c21517a0ccb1a,758.83,1439.84,0
3,dtf_0000004_240f3dae,0,op_03,28175.40,acc_o_ba23a2b955a79a8b,-443.88,-28619.28,acc_d_a3dd8504815ec133,770924.84,799100.24,0
4,dtf_0000005_f18939e7,0,op_03,86429.15,acc_o_d05a23079bd066c1,-670.85,-87100.01,acc_d_0d4880267e62d5c4,91.13,86520.29,0
5,dtf_0000006_64d9bfcb,0,op_05,787.04,acc_o_6d8f4254f3f09623,25.81,25.81,acc_d_d2528fd81d284bd6,0.00,0.00,0
6,dtf_0000007_12c897cc,0,op_05,771.06,acc_o_6ad1fd4988e085ad,572637.44,571866.37,acc_d_256141e7a7c7947e,629.98,1401.04,0
7,dtf_0000008_2017ee18,0,op_05,758.83,acc_o_ed558e2b5b5af5f7,115.21,-643.62,acc_d_89927ea80de472e6,0.00,758.83,0
8,dtf_0000009_7388f093,0,op_03,96800.66,acc_o_9cc0fd44ac3f05d3,-658.41,-97459.06,acc_d_a0682983bf40c207,70.76,96871.41,1
9,dtf_0000010_c0647632,0,op_05,649.32,acc_o_d185654afcfec4e4,93.32,93.32,acc_d_4a34c3d6fd957589,706.31,706.31,0


In [4]:
train.tail(10)

,id,period,operation,amount,origin_account,origin_balance_before,origin_balance_after,destination_account,destination_balance_before,destination_balance_after,fraud_flag
1290071,dtf_1290072_46b00111,105,op_03,19966.52,acc_o_d8efd2bdff5bb717,2361627.07,2341660.55,acc_d_3f26df6f3485043a,0.00,19966.52,0
1290072,dtf_1290073_6c3992ef,105,op_03,19858.60,acc_o_d5f9d0345b50b24f,4548420.25,4528561.65,acc_d_71863ac2c6b44aa8,0.00,19858.60,0
1290073,dtf_1290074_35ca31da,105,op_05,670.73,acc_o_ab7f0fcbb165fa6f,1473504.56,1472833.83,acc_d_b8e92c24c851b518,130042.07,130712.80,0
1290074,dtf_1290075_51b975c6,105,op_05,1032.99,acc_o_9565d761faee1db1,3704997.50,3703964.51,acc_d_975b5053917192e4,180870.68,181903.66,0
1290075,dtf_1290076_9d589c8e,105,op_05,646.23,acc_o_4c93de3b332fa53c,129778.06,129131.84,acc_d_58659476561a28b1,227103.29,227749.52,0
1290076,dtf_1290077_4b26aa01,105,op_05,667.47,acc_o_76b7f5595d9f2c9f,1688715.14,1688047.68,acc_d_661177be3e5965ca,203607.32,204274.77,0
1290077,dtf_1290078_07a32f90,105,op_03,19849.22,acc_o_4b132d78462d49d3,2593327.03,2573477.82,acc_d_d108b3d1928206ab,163529.43,183378.64,1
1290078,dtf_1290079_50f34347,105,op_05,633.99,acc_o_616eef85e0db107c,3527609.44,3526975.45,acc_d_3a161415514ba10f,276896.11,277530.10,0
1290079,dtf_1290080_d96382d2,105,op_05,711.40,acc_o_afaa6e5b6f55080f,2950643.39,2949931.99,acc_d_4b17724e353c1753,142609.30,143320.70,0
1290080,dtf_1290081_32be8ff7,105,op_03,89406.15,acc_o_a2df884812f3de57,623091.64,533685.47,acc_d_33e26a1331ec21eb,0.00,89406.15,1


In [5]:
train.sample(10)

,id,period,operation,amount,origin_account,origin_balance_before,origin_balance_after,destination_account,destination_balance_before,destination_balance_after,fraud_flag
1159465,dtf_1159466_73e621eb,92,op_03,20398.23,acc_o_38a3e1ece6419184,3689426.65,3669028.42,acc_d_b37cb80b89a0ba83,248203.90,268602.13,0
886805,dtf_0886806_124aa12c,58,op_05,712.36,acc_o_38ebafee7d495b05,3350433.60,3349721.24,acc_d_1ad4e07a514deef0,81419.99,82132.35,0
924722,dtf_0924723_4be5909c,63,op_03,29554.63,acc_o_29ea48713ebd43b5,2416326.02,2386771.39,acc_d_a606292e499bb3d1,0.00,29554.63,0
625102,dtf_0625103_e1732c8e,40,op_04,49306.60,acc_o_72c2ab6b7002645e,1532143.69,1581450.29,acc_d_20468c41d091bcd4,102492.34,102492.34,0
187627,dtf_0187628_2f9deec6,13,op_04,80765.24,acc_o_890081d2e9c442c6,256200.89,336966.13,acc_d_6cc6a8d8367715e1,21414.78,21414.78,0
1046773,dtf_1046774_cd76e88d,78,op_05,712.41,acc_o_71dab5de57903963,2695581.78,2694869.37,acc_d_375e1ae376bd2f11,223848.01,224560.41,0
825695,dtf_0825696_dbbb5699,50,op_03,23270.01,acc_o_ff5a64734737ae1a,2849192.47,2825922.46,acc_d_9057fa28c3e82298,0.00,23270.01,0
978644,dtf_0978645_d1168b2d,69,op_03,29175.71,acc_o_d5024bde81a694ae,1907447.39,1878271.67,acc_d_d2bbb5e9134da179,138551.89,167727.61,1
232704,dtf_0232705_320fb743,15,op_04,316906.13,acc_o_8baf1b68a6bee1fa,1023992.62,1340898.77,acc_d_cc1d99869f163863,11990.04,11990.04,0
1229314,dtf_1229315_31e00627,98,op_05,678.41,acc_o_032979419c0a3e5f,4743211.05,4742532.64,acc_d_67d0c7f974a154f7,193696.99,194375.40,0


In [6]:
train.nunique()

id                            1290081
period                            106
operation                           5
amount                         464079
origin_account                  13431
origin_balance_before         1245326
origin_balance_after          1247168
destination_account             15818
destination_balance_before     591925
destination_balance_after      651769
fraud_flag                          2
dtype: int64

In [7]:
train.groupby('operation')['fraud_flag'].agg(['sum', 'count', 'mean'])

,sum,count,mean
operation,,,
op_01,0,4087,0.000000
op_02,0,71876,0.000000
op_03,129542,415323,0.311907
op_04,0,305443,0.000000
op_05,0,493352,0.000000


In [8]:
# Et si on concentrait sur ce fameux op3
op3 = train[train['operation'] == 'op_03']
# Le delta de balance origine est-il cohérent avec le montant ?
op3['delta_origin'] = op3['origin_balance_after'] - op3['origin_balance_before']
op3['delta_check'] = op3['delta_origin'] + op3['amount']  # devrait être ~0 si cohérent

op3.groupby('fraud_flag')['delta_check'].describe()

,count,mean,std,min,25%,50%,75%,max
fraud_flag,,,,,,,,
0,285781.0,1835.598520,24031.962760,-0.02,-2.619345e-10,1.818989e-11,0.01,1717165.69
1,129542.0,2844.193884,15180.990821,-0.02,-1.891749e-10,2.182787e-11,0.01,559774.43


In [9]:
# Compter combien de transactions ont un delta_check "anormal" (au-delà d'un epsilon)
epsilon = 1.0  # tolérance pour arrondis flottants

op3['is_incoherent'] = op3['delta_check'].abs() > epsilon

pd.crosstab(op3['is_incoherent'], op3['fraud_flag'], normalize='index')

fraud_flag,0,1
is_incoherent,,
False,0.698266,0.301734
True,0.441110,0.558890


In [10]:
pd.crosstab(op3['is_incoherent'], op3['fraud_flag'], normalize='columns')

fraud_flag,0,1
is_incoherent,,
False,0.974641,0.92912
True,0.025359,0.07088


In [11]:
# ============================================================================
# 1. Fréquence d'utilisation du compte destination
# ============================================================================
dest_counts = op3['destination_account'].value_counts()
op3['dest_usage_count'] = op3['destination_account'].map(dest_counts)

print("Distribution de dest_usage_count selon fraud_flag:")
print(op3.groupby('fraud_flag')['dest_usage_count'].describe())

# ============================================================================
# 2. Compte destination "neuf" : solde à zéro avant réception
# ============================================================================
op3['dest_was_empty'] = op3['destination_balance_before'] == 0

print("\nTaux de fraude selon dest_was_empty:")
print(pd.crosstab(op3['dest_was_empty'], op3['fraud_flag'], normalize='index'))

print("\nRecall (parmi les fraudes, % avec dest_was_empty):")
print(pd.crosstab(op3['dest_was_empty'], op3['fraud_flag'], normalize='columns'))

Distribution de dest_usage_count selon fraud_flag:
               count       mean        std   min   25%   50%   75%    max
fraud_flag                                                               
0           285781.0  77.045615  16.815876   1.0  77.0  80.0  84.0  101.0
1           129542.0  80.584413   4.918119  64.0  77.0  80.0  84.0  101.0

Taux de fraude selon dest_was_empty:
fraud_flag             0         1
dest_was_empty                    
False           0.693817  0.306183
True            0.677630  0.322370

Recall (parmi les fraudes, % avec dest_was_empty):
fraud_flag             0         1
dest_was_empty                    
False           0.651796  0.634559
True            0.348204  0.365441


In [12]:
orig_counts = op3['origin_account'].value_counts()
op3['orig_usage_count'] = op3['origin_account'].map(orig_counts)

print(op3.groupby('fraud_flag')['orig_usage_count'].describe())

               count        mean        std  min   25%   50%    75%    max
fraud_flag                                                                
0           285781.0  100.037186  73.577627  1.0  45.0  82.0  137.0  480.0
1           129542.0   92.292554  73.839040  1.0  36.0  74.0  129.0  480.0


In [13]:
# 1. Ratio montant / solde origine (le compte est-il "vidé" ?)
op3['ratio_amount_balance'] = op3['amount'] / (op3['origin_balance_before'].abs() + 1)
print("Ratio amount/balance origine:")
print(op3.groupby('fraud_flag')['ratio_amount_balance'].describe())

# 2. Le compte origine devient-il négatif après la transaction ?
op3['orig_goes_negative'] = (op3['origin_balance_before'] >= 0) & (op3['origin_balance_after'] < 0)
print("\nTaux de fraude si le compte origine passe en négatif:")
print(pd.crosstab(op3['orig_goes_negative'], op3['fraud_flag'], normalize='index'))

# 3. Le montant est-il proche d'un seuil rond (pattern de fraude structurée) ?
op3['amount_rounded'] = (op3['amount'] % 1000 < 50) | (op3['amount'] % 1000 > 950)
print("\nTaux de fraude si montant proche d'un multiple de 1000:")
print(pd.crosstab(op3['amount_rounded'], op3['fraud_flag'], normalize='index'))

Ratio amount/balance origine:
               count      mean         std       min       25%       50%  \
fraud_flag                                                                 
0           285781.0  2.168029  409.355923  0.000007  0.005919  0.008605   
1           129542.0  1.300403  101.067548  0.001931  0.005980  0.008988   

                 75%            max  
fraud_flag                           
0           0.019637  198193.874786  
1           0.024570   29315.961661  

Taux de fraude si le compte origine passe en négatif:
fraud_flag                 0         1
orig_goes_negative                    
False               0.688057  0.311943
True                0.699043  0.300957

Taux de fraude si montant proche d'un multiple de 1000:
fraud_flag             0         1
amount_rounded                    
False           0.688165  0.311835
True            0.687518  0.312482


In [14]:
fraud_rate_per_origin = op3.groupby('origin_account')['fraud_flag'].mean()
print(fraud_rate_per_origin.value_counts(bins=5))

(0.2, 0.4]       6058
(0.8, 1.0]       1985
(-0.002, 0.2]    1935
(0.4, 0.6]        813
(0.6, 0.8]         96
Name: count, dtype: int64


In [15]:
print(op3.groupby('period')['fraud_flag'].agg(['sum', 'count', 'mean']).head(30))

         sum  count      mean
period                       
0       1177   2402  0.490008
1       1273   2556  0.498044
2       1265   2689  0.470435
3       1205   2652  0.454374
4       1158   2614  0.442999
5       1106   2501  0.442223
6       1015   2437  0.416496
7       1064   2505  0.424750
8       1115   2900  0.384483
9       1174   3221  0.364483
10      1270   3517  0.361103
11      1263   3664  0.344705
12      1259   3789  0.332278
13      1292   3872  0.333678
14      1248   4083  0.305658
15      1267   4284  0.295752
16      1183   4279  0.276466
17      1209   4382  0.275901
18      1245   4674  0.266367
19      1312   4736  0.277027
20      1219   4625  0.263568
21      1216   4472  0.271914
22      1233   4291  0.287346
23      1226   4106  0.298587
24      1326   3979  0.333250
25      1245   3898  0.319395
26      1231   3863  0.318664
27      1193   3859  0.309147
28      1229   3783  0.324874
29      1231   3804  0.323607


In [16]:
print("Train period range:", train['period'].min(), "-", train['period'].max())
print("Test period range:", test['period'].min(), "-", test['period'].max())

Train period range: 0 - 105
Test period range: 106 - 143


In [17]:
# Combien de comptes origine apparaissent à la fois dans train et test ?
common_accounts = set(train['origin_account']) & set(test['origin_account'])
print(f"Comptes en commun: {len(common_accounts)} / {test['origin_account'].nunique()} dans test")

# Combien de transactions par compte origine en moyenne ?
print(op3['origin_account'].value_counts().describe())

Comptes en commun: 10222 / 10253 dans test
count    10887.000000
mean        38.148526
std         47.634209
min          1.000000
25%          6.000000
50%         21.000000
75%         52.000000
max        480.000000
Name: count, dtype: float64


In [18]:
# ============================================================================
# Trier par compte + period pour respecter la chronologie
# ============================================================================
op3_sorted = op3.sort_values(['origin_account', 'period']).copy()

# Expanding mean DÉCALÉE (shift) pour éviter d'inclure la ligne courante
op3_sorted['orig_fraud_rate_hist'] = (
    op3_sorted.groupby('origin_account')['fraud_flag']
    .apply(lambda x: x.shift().expanding().mean())
    .reset_index(level=0, drop=True)
)

# Pour les comptes vus pour la 1ère fois (NaN), fallback = taux de fraude global op_03
global_rate = op3['fraud_flag'].mean()
op3_sorted['orig_fraud_rate_hist'] = op3_sorted['orig_fraud_rate_hist'].fillna(global_rate)

# Vérifions le pouvoir discriminant
print(op3_sorted.groupby('fraud_flag')['orig_fraud_rate_hist'].describe())

               count      mean       std  min       25%       50%       75%  \
fraud_flag                                                                    
0           285781.0  0.280014  0.130245  0.0  0.224138  0.280000  0.333333   
1           129542.0  0.346156  0.237712  0.0  0.235294  0.291667  0.351351   

            max  
fraud_flag       
0           1.0  
1           1.0  


In [19]:
# Taux de fraude par compte sur TOUT le train (pas de leakage car test est dans le futur)
orig_fraud_rate_train = train.groupby('origin_account')['fraud_flag'].mean()
test['orig_fraud_rate_hist'] = test['origin_account'].map(orig_fraud_rate_train).fillna(global_rate)

In [20]:
# Compter combien d'observations passées on a pour chaque ligne
op3_sorted['orig_n_obs_hist'] = (
    op3_sorted.groupby('origin_account').cumcount()
)

# Filtrer sur les comptes avec un historique suffisant (ex: au moins 10 transactions passées)
mask = op3_sorted['orig_n_obs_hist'] >= 10
print(op3_sorted[mask].groupby('fraud_flag')['orig_fraud_rate_hist'].describe())

               count      mean       std  min       25%       50%       75%  \
fraud_flag                                                                    
0           229634.0  0.280216  0.080281  0.0  0.234043  0.279221  0.324324   
1            99945.0  0.318046  0.177256  0.0  0.238095  0.285714  0.333333   

            max  
fraud_flag       
0           0.8  
1           1.0  


In [21]:
high_risk = op3_sorted['orig_fraud_rate_hist'] > 0.4

print("Croisement high_risk x is_incoherent -> taux de fraude (normalize='index'):")
print(pd.crosstab([high_risk, op3_sorted['is_incoherent']], op3_sorted['fraud_flag'], normalize='index'))

print("\nCroisement high_risk x is_incoherent -> recall (normalize='columns'):")
print(pd.crosstab([high_risk, op3_sorted['is_incoherent']], op3_sorted['fraud_flag'], normalize='columns'))

print("\nEffectifs (count) de chaque combinaison:")
print(pd.crosstab([high_risk, op3_sorted['is_incoherent']], op3_sorted['fraud_flag']))

Croisement high_risk x is_incoherent -> taux de fraude (normalize='index'):
fraud_flag                                 0         1
orig_fraud_rate_hist is_incoherent                    
False                False          0.708804  0.291196
                     True           0.649652  0.350348
True                 False          0.606282  0.393718
                     True           0.156470  0.843530

Croisement high_risk x is_incoherent -> recall (normalize='columns'):
fraud_flag                                 0         1
orig_fraud_rate_hist is_incoherent                    
False                False          0.887652  0.804496
                     True           0.021555  0.025644
True                 False          0.086990  0.124624
                     True           0.003804  0.045236

Effectifs (count) de chaque combinaison:
fraud_flag                               0       1
orig_fraud_rate_hist is_incoherent                
False                False          253674  10421

In [22]:
train['fraud_flag'].value_counts()

fraud_flag
0    1160539
1     129542
Name: count, dtype: int64

# Construction

### Feature engeneering

In [23]:
def build_features(df, fraud_rate_map=None, is_train=True, orig_counts_train=None):
    """
    Construit les features sur op_03.
    fraud_rate_map     : dict {origin_account: taux_fraude} calculé sur train, réutilisé pour test.
    orig_counts_train  : Series des comptages par compte origine, calculée sur train, réutilisée pour test.
    """
    df = df.copy()
    epsilon = 1.0

    # --- Cohérence comptable côté origine ---
    df['delta_check'] = (
        df['origin_balance_after'] - df['origin_balance_before'] + df['amount']
    )
    df['is_incoherent'] = (df['delta_check'].abs() > epsilon).astype(int)

    # --- Ratio montant / solde ---
    df['ratio_amount_balance'] = df['amount'] / (df['origin_balance_before'].abs() + 1)

    # --- Cohérence comptable côté destination ---
    df['delta_dest'] = (
        df['destination_balance_after'] - df['destination_balance_before'] - df['amount']
    )
    df['is_incoherent_dest'] = (df['delta_dest'].abs() > epsilon).astype(int)

    # --- Taux de fraude historique par compte origine ---
    if is_train:
        global_rate = df['fraud_flag'].mean()

        df = df.sort_values(['origin_account', 'period'])
        df['orig_fraud_rate_hist'] = (
            df.groupby('origin_account')['fraud_flag']
            .apply(lambda x: x.shift().expanding().mean())
            .reset_index(level=0, drop=True)
        )
        df['orig_fraud_rate_hist'] = df['orig_fraud_rate_hist'].fillna(global_rate)
        df['orig_n_obs_hist'] = df.groupby('origin_account').cumcount()

        fraud_rate_map = df.groupby('origin_account')['fraud_flag'].mean().to_dict()
        fraud_rate_map['__global__'] = global_rate
        orig_counts_train = df['origin_account'].value_counts()

    else:
        global_rate_value = fraud_rate_map['__global__']
        df['orig_fraud_rate_hist'] = df['origin_account'].map(fraud_rate_map).fillna(global_rate_value)
        df['orig_n_obs_hist'] = df['origin_account'].map(orig_counts_train).fillna(0)

    # --- Interaction : compte à risque ET incohérent ---
    df['high_risk'] = (df['orig_fraud_rate_hist'] > 0.4).astype(int)
    df['high_risk_incoherent'] = df['high_risk'] * df['is_incoherent']

    return df, fraud_rate_map, orig_counts_train

In [24]:
## deuxième fonction

def build_features(df, fraud_rate_map=None, is_train=True, orig_counts_train=None, k=15):
    """
    Construit les features sur op_03.
    k : poids du prior pour le lissage bayésien (plus k est grand, plus on tire vers le taux global)
    """
    df = df.copy()
    epsilon = 1.0

    # --- Cohérence comptable côté origine ---
    df['delta_check'] = (
        df['origin_balance_after'] - df['origin_balance_before'] + df['amount']
    )
    df['is_incoherent'] = (df['delta_check'].abs() > epsilon).astype(int)

    # --- Ratio montant / solde ---
    df['ratio_amount_balance'] = df['amount'] / (df['origin_balance_before'].abs() + 1)

    # --- Cohérence comptable côté destination ---
    df['delta_dest'] = (
        df['destination_balance_after'] - df['destination_balance_before'] - df['amount']
    )
    df['is_incoherent_dest'] = (df['delta_dest'].abs() > epsilon).astype(int)

    # --- Taux de fraude historique par compte origine ---
    if is_train:
        global_rate = df['fraud_flag'].mean()

        df = df.sort_values(['origin_account', 'period'])

        # Expanding SUM et COUNT décalés (au lieu de mean directement, pour le lissage)
        grouped = df.groupby('origin_account')['fraud_flag']
        df['orig_fraud_sum_hist'] = grouped.apply(lambda x: x.shift().expanding().sum()).reset_index(level=0, drop=True)
        df['orig_n_obs_hist'] = grouped.apply(lambda x: x.shift().expanding().count()).reset_index(level=0, drop=True)

        df['orig_fraud_sum_hist'] = df['orig_fraud_sum_hist'].fillna(0)
        df['orig_n_obs_hist'] = df['orig_n_obs_hist'].fillna(0)

        # Lissage bayésien : (sum + global_rate * k) / (n_obs + k)
        df['orig_fraud_rate_hist'] = (
            (df['orig_fraud_sum_hist'] + global_rate * k) / (df['orig_n_obs_hist'] + k)
        )

        # Stocker sum/count par compte (pas juste la moyenne) pour réutiliser sur test
        fraud_rate_map = {
            'sum': df.groupby('origin_account')['fraud_flag'].sum().to_dict(),
            'count': df.groupby('origin_account')['fraud_flag'].count().to_dict(),
            '__global__': global_rate
        }
        orig_counts_train = df['origin_account'].value_counts()

    else:
        global_rate_value = fraud_rate_map['__global__']
        sum_map = fraud_rate_map['sum']
        count_map = fraud_rate_map['count']

        df['orig_fraud_sum_hist'] = df['origin_account'].map(sum_map).fillna(0)
        df['orig_n_obs_hist'] = df['origin_account'].map(count_map).fillna(0)

        # Même formule de lissage, basée sur TOUT l'historique train (le test est dans le futur)
        df['orig_fraud_rate_hist'] = (
            (df['orig_fraud_sum_hist'] + global_rate_value * k) / (df['orig_n_obs_hist'] + k)
        )

    # --- Interaction : compte à risque ET incohérent ---
    df['high_risk'] = (df['orig_fraud_rate_hist'] > 0.4).astype(int)
    df['high_risk_incoherent'] = df['high_risk'] * df['is_incoherent']

    return df, fraud_rate_map, orig_counts_train

In [25]:
# --- Isoler op_03 ---
train_op3 = train[train['operation'] == 'op_03'].copy()
test_op3 = test[test['operation'] == 'op_03'].copy()
test_other = test[test['operation'] != 'op_03'].copy()

print(f"Train op_03: {train_op3.shape[0]} lignes")
print(f"Test op_03: {test_op3.shape[0]} lignes")
print(f"Test autres opérations (fraud=0 direct): {test_other.shape[0]} lignes")

# --- Feature engineering ---
train_feat, fraud_rate_map, orig_counts_train = build_features(train_op3, is_train=True)
test_feat, _, _ = build_features(
    test_op3, fraud_rate_map=fraud_rate_map, is_train=False, orig_counts_train=orig_counts_train
)

feature_cols = [ # on retire is_incoherent_dest et high_risk
    'amount', 'origin_balance_before', 'origin_balance_after',
    'destination_balance_before', 'destination_balance_after',
    'period', 'is_incoherent', 'ratio_amount_balance',
    'orig_fraud_rate_hist', 'orig_n_obs_hist', 'high_risk_incoherent'
]


X = train_feat[feature_cols]
y = train_feat['fraud_flag']
X_test = test_feat[feature_cols]

print(f"\n✅ X train: {X.shape}, X test: {X_test.shape}")

Train op_03: 415323 lignes
Test op_03: 154005 lignes
Test autres opérations (fraud=0 direct): 276095 lignes

✅ X train: (415323, 11), X test: (154005, 11)


In [26]:
# Split temporel : on simule la même logique que train/test (derniers period = validation)
period_threshold = train_feat['period'].quantile(0.8)

train_mask = train_feat['period'] <= period_threshold
val_mask = train_feat['period'] > period_threshold

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Taux fraude train: {y_train.mean():.3f}, Taux fraude val: {y_val.mean():.3f}")

Train: (332335, 11), Val: (82988, 11)
Taux fraude train: 0.313, Taux fraude val: 0.307


In [27]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'average_precision',
    'boosting_type': 'gbdt',
    'scale_pos_weight': scale_pos_weight,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 10,
    'verbose': -1,
    'seed': 42
}

model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'val'],
    num_boost_round=1000,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

scale_pos_weight: 2.193
Training until validation scores don't improve for 50 rounds
[50]	train's average_precision: 0.472394	val's average_precision: 0.356006
[100]	train's average_precision: 0.486019	val's average_precision: 0.360993
[150]	train's average_precision: 0.50316	val's average_precision: 0.36243
[200]	train's average_precision: 0.516527	val's average_precision: 0.36182
Early stopping, best iteration is:
[164]	train's average_precision: 0.506187	val's average_precision: 0.362794


In [28]:
y_pred_val = model.predict(X_val, num_iteration=model.best_iteration)
pr_auc = average_precision_score(y_val, y_pred_val)
print(f"\n🎯 PR-AUC sur validation temporelle: {pr_auc:.4f}")

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\n📊 Importance des features:")
print(importance)


🎯 PR-AUC sur validation temporelle: 0.3628

📊 Importance des features:
                       feature     importance
8         orig_fraud_rate_hist  231239.926360
0                       amount  197773.103968
1        origin_balance_before   41577.501236
9              orig_n_obs_hist   38764.210504
2         origin_balance_after   29551.375178
5                       period   12461.351112
7         ratio_amount_balance   11719.941786
3   destination_balance_before   10480.102017
4    destination_balance_after    9906.736116
10        high_risk_incoherent    3790.185370
6                is_incoherent    1297.565626


In [29]:
scale_pos_weight_full = (y == 0).sum() / (y == 1).sum()
params['scale_pos_weight'] = scale_pos_weight_full

train_data_full = lgb.Dataset(X, label=y)

model_final = lgb.train(
    params,
    train_data_full,
    num_boost_round=model.best_iteration  # nombre d'arbres optimal trouvé sur la validation temporelle
)

print(f"✅ Modèle final entraîné ({model.best_iteration} arbres)")

✅ Modèle final entraîné (164 arbres)


In [30]:
test_feat['target'] = model_final.predict(X_test)
test_other['target'] = 0.0

submission = pd.concat([
    test_feat[['id', 'target']],
    test_other[['id', 'target']]
])

submission = submission.set_index('id').loc[test['id']].reset_index()

print(submission.head())
print(f"\n✅ Shape finale: {submission.shape} (doit matcher test: {test.shape[0]})")

                     id    target
0  dtf_0000001_08a8a524  0.000000
1  dtf_0000002_ae0d3769  0.475705
2  dtf_0000003_843bab7c  0.485788
3  dtf_0000004_91643844  0.000000
4  dtf_0000005_17bd9a08  0.492610

✅ Shape finale: (430100, 2) (doit matcher test: 430100)


In [31]:
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv créé !")

✅ submission.csv créé !
